In [ ]:
import os
import warnings
import logging
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import optuna
from optuna.samplers import TPESampler
import shap
import traceback
from scipy import stats
from typing import Dict, Tuple, Any, Optional

warnings.filterwarnings("ignore")

# Configure logging
logging.basicConfig(
    level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s"
)
logger = logging.getLogger(__name__)

# Suppress Optuna logging
optuna.logging.set_verbosity(optuna.logging.WARNING)

RANDOM_SEED = 2025
np.random.seed(RANDOM_SEED)


class RandomForestGlucosePredictor:
    """RandomForest Glucose Predictor - Bayesian Optimization Version"""

    def __init__(self, random_state: int = RANDOM_SEED):
        self.random_state = random_state
        self.model_30min = None
        self.model_60min = None
        self.feature_names = None
        self.best_params_30min = None
        self.best_params_60min = None
        self.results_dir = "results"
        os.makedirs(self.results_dir, exist_ok=True)

    def load_data(self) -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
        """Load preprocessed data"""
        logger.info("Loading data...")
        try:
            train_df = pd.read_csv("train_enhanced_processed.csv")
            val_df = pd.read_csv("val_enhanced_processed.csv")
            test_df = pd.read_csv("test_enhanced_processed.csv")

            logger.info(f"Training set: {train_df.shape}")
            logger.info(f"Validation set: {val_df.shape}")
            logger.info(f"Test set: {test_df.shape}")

            for df in [train_df, val_df, test_df]:
                if "timestamp" in df.columns:
                    df["timestamp"] = pd.to_datetime(df["timestamp"])

            return train_df, val_df, test_df
        except FileNotFoundError as e:
            logger.error(f"Data file not found: {e}")
            raise

    def prepare_features(
        self, df: pd.DataFrame, target: str = "glucose_30min"
    ) -> Tuple[pd.DataFrame, pd.Series]:
        """Prepare features and target variable"""
        exclude_cols = ["patient_id", "timestamp", "glucose_30min", "glucose_60min"]
        feature_cols = [col for col in df.columns if col not in exclude_cols]

        X = df[feature_cols].copy()
        y = df[target].copy()

        valid_mask = ~(X.isna().any(axis=1) | y.isna())
        X = X[valid_mask]
        y = y[valid_mask]

        if self.feature_names is None:
            self.feature_names = feature_cols

        logger.info(f"Target variable: {target}")
        logger.info(f"Number of features: {len(feature_cols)}")
        logger.info(f"Number of valid samples: {len(X)}")

        return X, y

    def objective(
        self,
        trial: optuna.Trial,
        X_train: pd.DataFrame,
        y_train: pd.Series,
        X_val: pd.DataFrame,
        y_val: pd.Series,
    ) -> float:
        """Optuna objective function for Bayesian optimization"""
        params = {
            "random_state": self.random_state,
            "n_jobs": -1,
            "n_estimators": trial.suggest_int("n_estimators", 100, 2000),
            "max_depth": trial.suggest_int("max_depth", 3, 50),
            "min_samples_split": trial.suggest_int("min_samples_split", 2, 100),
            "min_samples_leaf": trial.suggest_int("min_samples_leaf", 1, 50),
            "max_features": trial.suggest_categorical(
                "max_features", ["sqrt", "log2", None]
            ),
            "min_impurity_decrease": trial.suggest_float(
                "min_impurity_decrease", 0.0, 0.5
            ),
            "bootstrap": trial.suggest_categorical("bootstrap", [True, False]),
            "oob_score": False,
            "max_leaf_nodes": trial.suggest_int("max_leaf_nodes", 10, 1000),
            "min_weight_fraction_leaf": trial.suggest_float(
                "min_weight_fraction_leaf", 0.0, 0.1
            ),
            "ccp_alpha": trial.suggest_float("ccp_alpha", 0.0, 0.1),
        }

        # Only set max_samples when bootstrap=True
        if params["bootstrap"]:
            params["max_samples"] = trial.suggest_float("max_samples", 0.5, 1.0)

        # Handle max_depth None case
        if params["max_depth"] == 50:
            if trial.suggest_categorical("use_none_max_depth", [True, False]):
                params["max_depth"] = None

        model = RandomForestRegressor(**params)
        model.fit(X_train, y_train)
        y_pred = model.predict(X_val)
        rmse = np.sqrt(mean_squared_error(y_val, y_pred))

        return rmse

    def tune_hyperparameters(
        self,
        X_train: pd.DataFrame,
        y_train: pd.Series,
        X_val: pd.DataFrame,
        y_val: pd.Series,
        n_trials: int = 200,
    ) -> Tuple[RandomForestRegressor, Dict[str, Any], pd.DataFrame]:
        """Tune hyperparameters using Bayesian optimization (Optuna)"""
        logger.info("=" * 60)
        logger.info("Starting Bayesian hyperparameter optimization...")
        logger.info(f"Number of optimization trials: {n_trials}")
        logger.info("=" * 60)

        study = optuna.create_study(
            direction="minimize",
            sampler=TPESampler(seed=self.random_state),
            study_name="RandomForest_Glucose_Prediction",
        )

        study.optimize(
            lambda trial: self.objective(trial, X_train, y_train, X_val, y_val),
            n_trials=n_trials,
            show_progress_bar=True,
        )

        best_params = study.best_params
        best_rmse = study.best_value

        logger.info("\nBest parameters found:")
        for param, value in best_params.items():
            logger.info(f"  {param}: {value}")
        logger.info(f"\nBest validation RMSE: {best_rmse:.4f} mg/dL")

        # Train final model with best parameters
        final_params = {
            "random_state": self.random_state,
            "n_jobs": -1,
            "oob_score": False,
        }

        bootstrap_value = None
        for key, value in best_params.items():
            if key == "use_none_max_depth":
                continue
            if key == "bootstrap":
                bootstrap_value = value
            final_params[key] = value

        # Remove max_samples if bootstrap=False
        if bootstrap_value is False and "max_samples" in final_params:
            del final_params["max_samples"]

        # Handle max_depth None case
        if "use_none_max_depth" in best_params and best_params["use_none_max_depth"]:
            final_params["max_depth"] = None

        best_model = RandomForestRegressor(**final_params)
        best_model.fit(X_train, y_train)

        optimization_history = pd.DataFrame(
            {
                "trial": range(len(study.trials)),
                "value": [trial.value for trial in study.trials],
            }
        )

        return best_model, best_params, optimization_history

    def train_models(
        self, train_df: pd.DataFrame, val_df: pd.DataFrame, n_trials: int = 200
    ) -> None:
        """Train 30-minute and 60-minute prediction models"""
        # Train 30-minute model
        logger.info("\n" + "=" * 60)
        logger.info("Training 30-minute glucose prediction model")
        logger.info("=" * 60)

        X_train_30, y_train_30 = self.prepare_features(train_df, "glucose_30min")
        X_val_30, y_val_30 = self.prepare_features(val_df, "glucose_30min")

        self.model_30min, self.best_params_30min, history_30 = (
            self.tune_hyperparameters(
                X_train_30, y_train_30, X_val_30, y_val_30, n_trials=n_trials
            )
        )

        history_30.to_csv(
            f"{self.results_dir}/optimization_history_30min.csv", index=False
        )

        val_pred_30 = self.model_30min.predict(X_val_30)
        val_rmse_30 = np.sqrt(mean_squared_error(y_val_30, val_pred_30))
        val_mae_30 = mean_absolute_error(y_val_30, val_pred_30)
        val_r2_30 = r2_score(y_val_30, val_pred_30)

        logger.info("\n30-minute model validation set performance:")
        logger.info(f"  RMSE: {val_rmse_30:.4f} mg/dL")
        logger.info(f"  MAE: {val_mae_30:.4f} mg/dL")
        logger.info(f"  R²: {val_r2_30:.4f}")

        # Train 60-minute model
        logger.info("\n" + "=" * 60)
        logger.info("Training 60-minute glucose prediction model")
        logger.info("=" * 60)

        X_train_60, y_train_60 = self.prepare_features(train_df, "glucose_60min")
        X_val_60, y_val_60 = self.prepare_features(val_df, "glucose_60min")

        self.model_60min, self.best_params_60min, history_60 = (
            self.tune_hyperparameters(
                X_train_60, y_train_60, X_val_60, y_val_60, n_trials=n_trials
            )
        )

        history_60.to_csv(
            f"{self.results_dir}/optimization_history_60min.csv", index=False
        )

        val_pred_60 = self.model_60min.predict(X_val_60)
        val_rmse_60 = np.sqrt(mean_squared_error(y_val_60, val_pred_60))
        val_mae_60 = mean_absolute_error(y_val_60, val_pred_60)
        val_r2_60 = r2_score(y_val_60, val_pred_60)

        logger.info("\n60-minute model validation set performance:")
        logger.info(f"  RMSE: {val_rmse_60:.4f} mg/dL")
        logger.info(f"  MAE: {val_mae_60:.4f} mg/dL")
        logger.info(f"  R²: {val_r2_60:.4f}")

        params_df = pd.DataFrame(
            {"30min": self.best_params_30min, "60min": self.best_params_60min}
        )
        params_df.to_csv(f"{self.results_dir}/best_hyperparameters.csv")
        logger.info(
            f"\nBest hyperparameters saved to {self.results_dir}/best_hyperparameters.csv"
        )

        self.plot_optimization_history(history_30, history_60)

    def plot_optimization_history(
        self, history_30: pd.DataFrame, history_60: pd.DataFrame
    ) -> None:
        """Plot Bayesian optimization history"""
        logger.info("\nGenerating optimization history plots...")

        fig, axes = plt.subplots(1, 2, figsize=(16, 6), dpi=300)

        axes[0].plot(history_30["trial"], history_30["value"], "b-", alpha=0.6)
        axes[0].plot(
            history_30["trial"],
            history_30["value"].cummin(),
            "r-",
            linewidth=2,
            label="Best RMSE",
        )
        axes[0].set_xlabel("Trial", fontsize=12)
        axes[0].set_ylabel("Validation RMSE (mg/dL)", fontsize=12)
        axes[0].set_title(
            "30-min Model - Bayesian Optimization History",
            fontsize=14,
            fontweight="bold",
        )
        axes[0].legend(fontsize=10)
        axes[0].grid(True, alpha=0.3)

        axes[1].plot(history_60["trial"], history_60["value"], "b-", alpha=0.6)
        axes[1].plot(
            history_60["trial"],
            history_60["value"].cummin(),
            "r-",
            linewidth=2,
            label="Best RMSE",
        )
        axes[1].set_xlabel("Trial", fontsize=12)
        axes[1].set_ylabel("Validation RMSE (mg/dL)", fontsize=12)
        axes[1].set_title(
            "60-min Model - Bayesian Optimization History",
            fontsize=14,
            fontweight="bold",
        )
        axes[1].legend(fontsize=10)
        axes[1].grid(True, alpha=0.3)

        plt.tight_layout()
        plt.savefig(
            f"{self.results_dir}/optimization_history.png", dpi=300, bbox_inches="tight"
        )
        plt.show()
        plt.close()

        logger.info("Optimization history plots saved")

    def evaluate_on_test(self, test_df: pd.DataFrame) -> Dict[str, Dict[str, Any]]:
        """Evaluate on test set"""
        logger.info("\n" + "=" * 60)
        logger.info("Test set evaluation")
        logger.info("=" * 60)

        results = {}

        # 30-minute prediction
        X_test_30, y_test_30 = self.prepare_features(test_df, "glucose_30min")
        pred_30 = self.model_30min.predict(X_test_30)

        rmse_30 = np.sqrt(mean_squared_error(y_test_30, pred_30))
        mae_30 = mean_absolute_error(y_test_30, pred_30)
        r2_30 = r2_score(y_test_30, pred_30)

        logger.info("\n30-minute prediction:")
        logger.info(f"  RMSE: {rmse_30:.4f} mg/dL")
        logger.info(f"  MAE: {mae_30:.4f} mg/dL")
        logger.info(f"  R²: {r2_30:.4f}")

        results["30min"] = {
            "y_true": y_test_30,
            "y_pred": pred_30,
            "rmse": rmse_30,
            "mae": mae_30,
            "r2": r2_30,
            "X": X_test_30,
        }

        # 60-minute prediction
        X_test_60, y_test_60 = self.prepare_features(test_df, "glucose_60min")
        pred_60 = self.model_60min.predict(X_test_60)

        rmse_60 = np.sqrt(mean_squared_error(y_test_60, pred_60))
        mae_60 = mean_absolute_error(y_test_60, pred_60)
        r2_60 = r2_score(y_test_60, pred_60)

        logger.info("\n60-minute prediction:")
        logger.info(f"  RMSE: {rmse_60:.4f} mg/dL")
        logger.info(f"  MAE: {mae_60:.4f} mg/dL")
        logger.info(f"  R²: {r2_60:.4f}")

        results["60min"] = {
            "y_true": y_test_60,
            "y_pred": pred_60,
            "rmse": rmse_60,
            "mae": mae_60,
            "r2": r2_60,
            "X": X_test_60,
        }

        for horizon in ["30min", "60min"]:
            result_df = pd.DataFrame(
                {
                    "y_true": results[horizon]["y_true"].values,
                    "y_pred": results[horizon]["y_pred"],
                }
            )
            result_df.to_csv(
                f"{self.results_dir}/predictions_{horizon}.csv", index=False
            )

        logger.info(f"\nPrediction results saved to {self.results_dir}/")

        return results

    def plot_predictions(self, results: Dict[str, Dict[str, Any]]) -> None:
        """Plot prediction comparison (4-in-1 plot)"""
        logger.info("\nGenerating prediction comparison plots...")

        for horizon, data in results.items():
            y_true = data["y_true"].values
            y_pred = data["y_pred"]

            plot_samples = min(500, len(y_true))
            indices = np.linspace(0, len(y_true) - 1, plot_samples, dtype=int)

            fig, axes = plt.subplots(2, 2, figsize=(16, 12), dpi=300)

            # 1. Time series comparison
            axes[0, 0].plot(
                indices,
                y_true[indices],
                "b-",
                label="Ground Truth",
                alpha=0.7,
                linewidth=1.5,
            )
            axes[0, 0].plot(
                indices,
                y_pred[indices],
                "r--",
                label="Prediction",
                alpha=0.7,
                linewidth=1.5,
            )
            axes[0, 0].fill_between(
                indices, y_true[indices], y_pred[indices], alpha=0.2, color="gray"
            )
            axes[0, 0].set_xlabel("Sample Index", fontsize=11)
            axes[0, 0].set_ylabel("Glucose Level (mg/dL)", fontsize=11)
            axes[0, 0].set_title(
                f"{horizon} Glucose Prediction - Time Series Comparison",
                fontsize=13,
                fontweight="bold",
            )
            axes[0, 0].legend(fontsize=10)
            axes[0, 0].grid(True, alpha=0.3)

            # 2. Scatter plot
            axes[0, 1].scatter(y_true, y_pred, alpha=0.3, s=10, c="steelblue")

            min_val = min(y_true.min(), y_pred.min())
            max_val = max(y_true.max(), y_pred.max())
            axes[0, 1].plot(
                [min_val, max_val],
                [min_val, max_val],
                "r--",
                linewidth=2,
                label="Perfect Prediction",
                alpha=0.8,
            )

            axes[0, 1].set_xlabel("True Glucose Level (mg/dL)", fontsize=11)
            axes[0, 1].set_ylabel("Predicted Glucose Level (mg/dL)", fontsize=11)
            axes[0, 1].set_title(
                f"{horizon} Glucose Prediction - Scatter Plot",
                fontsize=13,
                fontweight="bold",
            )
            axes[0, 1].legend(fontsize=10)
            axes[0, 1].grid(True, alpha=0.3)

            textstr = f"RMSE: {data['rmse']:.2f} mg/dL\nMAE: {data['mae']:.2f} mg/dL\nR²: {data['r2']:.4f}"
            axes[0, 1].text(
                0.05,
                0.95,
                textstr,
                transform=axes[0, 1].transAxes,
                verticalalignment="top",
                bbox=dict(boxstyle="round", facecolor="wheat", alpha=0.7),
                fontsize=10,
            )

            # 3. Residual plot
            residuals = y_true - y_pred
            axes[1, 0].scatter(y_pred, residuals, alpha=0.3, s=10, c="coral")
            axes[1, 0].axhline(y=0, color="r", linestyle="--", linewidth=2)
            axes[1, 0].set_xlabel("Predicted Glucose Level (mg/dL)", fontsize=11)
            axes[1, 0].set_ylabel("Residual (mg/dL)", fontsize=11)
            axes[1, 0].set_title(
                f"{horizon} Glucose Prediction - Residual Plot",
                fontsize=13,
                fontweight="bold",
            )
            axes[1, 0].grid(True, alpha=0.3)

            residual_mean = np.mean(residuals)
            residual_std = np.std(residuals)
            axes[1, 0].text(
                0.05,
                0.95,
                f"Mean: {residual_mean:.2f}\nStd Dev: {residual_std:.2f}",
                transform=axes[1, 0].transAxes,
                verticalalignment="top",
                bbox=dict(boxstyle="round", facecolor="lightblue", alpha=0.7),
                fontsize=10,
            )

            # 4. Residual distribution
            axes[1, 1].hist(
                residuals, bins=50, alpha=0.7, color="blue", edgecolor="black"
            )
            axes[1, 1].axvline(
                x=0, color="r", linestyle="--", linewidth=2, label="Zero Residual"
            )

            mu, sigma = stats.norm.fit(residuals)
            x = np.linspace(residuals.min(), residuals.max(), 100)
            p = stats.norm.pdf(x, mu, sigma)
            ax2 = axes[1, 1].twinx()
            ax2.plot(x, p, "r-", linewidth=2, label="Normal Distribution Fit")
            ax2.set_ylabel("Probability Density", fontsize=10)

            axes[1, 1].set_xlabel("Residual (mg/dL)", fontsize=11)
            axes[1, 1].set_ylabel("Frequency", fontsize=11)
            axes[1, 1].set_title(
                f"{horizon} Glucose Prediction - Residual Distribution",
                fontsize=13,
                fontweight="bold",
            )
            axes[1, 1].legend(loc="upper left", fontsize=9)
            ax2.legend(loc="upper right", fontsize=9)
            axes[1, 1].grid(True, alpha=0.3)

            plt.tight_layout()
            plt.savefig(
                f"{self.results_dir}/prediction_comparison_{horizon}.png",
                dpi=300,
                bbox_inches="tight",
            )
            plt.show()
            plt.close()

            logger.info(f"{horizon} prediction curves saved")

    def plot_time_series_by_patient(
        self,
        results: Dict[str, Dict[str, Any]],
        test_df: pd.DataFrame,
        num_patients: int = 2,
    ) -> None:
        """Plot time series by patient (if timestamp and patient_id are available)"""
        if "timestamp" not in test_df.columns or "patient_id" not in test_df.columns:
            logger.warning(
                "Missing timestamp or patient_id columns, skipping patient-level time series plots"
            )
            return

        logger.info("\nGenerating patient-level time series plots...")

        patient_ids = test_df["patient_id"].unique()[:num_patients]

        for horizon, data in results.items():
            _ = "glucose_30min" if horizon == "30min" else "glucose_60min"

            for patient_id in patient_ids:
                try:
                    # Get patient data
                    patient_mask = test_df["patient_id"] == patient_id
                    patient_data = test_df[patient_mask].copy()

                    if len(patient_data) < 10:
                        logger.info(f"Skipping patient {patient_id}: insufficient data")
                        continue

                    # **KEY FIX**: Match indices properly
                    valid_indices = patient_data.index.intersection(
                        data["y_true"].index
                    )

                    if len(valid_indices) == 0:
                        logger.warning(f"No matching indices for patient {patient_id}")
                        continue

                    # Get aligned true and predicted values
                    y_true_patient = data["y_true"].loc[valid_indices].values

                    # Get predictions using index positions
                    pred_positions = [
                        data["y_true"].index.get_loc(idx) for idx in valid_indices
                    ]
                    y_pred_patient = data["y_pred"][pred_positions]

                    # Get timestamps
                    timestamps = patient_data.loc[valid_indices, "timestamp"].values

                    # Plotting
                    fig, ax = plt.subplots(figsize=(16, 6), dpi=300)

                    ax.plot(
                        timestamps,
                        y_true_patient,
                        "b-",
                        label="Ground Truth",
                        linewidth=2,
                        alpha=0.7,
                    )
                    ax.plot(
                        timestamps,
                        y_pred_patient,
                        "r--",
                        label="Prediction",
                        linewidth=2,
                        alpha=0.7,
                    )
                    ax.fill_between(
                        timestamps,
                        y_true_patient,
                        y_pred_patient,
                        alpha=0.2,
                        color="gray",
                    )

                    ax.axhspan(
                        70,
                        180,
                        alpha=0.1,
                        color="green",
                        label="Target Range (70-180 mg/dL)",
                    )

                    ax.set_xlabel("Time", fontsize=12)
                    ax.set_ylabel("Glucose Level (mg/dL)", fontsize=12)
                    ax.set_title(
                        f"{horizon} Glucose Prediction - Patient {patient_id} Time Series",
                        fontsize=14,
                        fontweight="bold",
                        pad=20,
                    )
                    ax.legend(fontsize=10, loc="best")
                    ax.grid(True, alpha=0.3)

                    plt.xticks(rotation=45)
                    plt.tight_layout()
                    plt.savefig(
                        f"{self.results_dir}/time_series_{horizon}_patient_{patient_id}.png",
                        dpi=300,
                        bbox_inches="tight",
                    )
                    plt.show()
                    plt.close()

                    logger.info(f"Patient {patient_id} time series plot saved")

                except Exception as e:
                    logger.warning(
                        f"Failed to plot {horizon} time series for patient {patient_id}: {e}"
                    )
                    traceback.print_exc()
                    continue

    def shap_analysis(
        self, results: Dict[str, Dict[str, Any]], max_samples: int = 100
    ) -> None:
        """SHAP interpretability analysis"""
        logger.info("\n" + "=" * 60)
        logger.info("SHAP Interpretability Analysis")
        logger.info("=" * 60)

        for horizon, data in results.items():
            logger.info(f"\nAnalyzing {horizon} prediction model...")

            model = self.model_30min if horizon == "30min" else self.model_60min
            X = data["X"]

            if len(X) > max_samples:
                sample_indices = np.random.choice(len(X), max_samples, replace=False)
                X_sample = X.iloc[sample_indices]
            else:
                X_sample = X

            try:
                explainer = shap.TreeExplainer(model)
                shap_values = explainer.shap_values(X_sample)

                # 1. Summary Plot
                plt.figure(figsize=(12, 8), dpi=300)
                shap.summary_plot(shap_values, X_sample, show=False, max_display=20)
                plt.title(
                    f"{horizon} Glucose Prediction - SHAP Summary Plot",
                    pad=20,
                    fontsize=14,
                    fontweight="bold",
                )
                plt.tight_layout()
                plt.savefig(
                    f"{self.results_dir}/shap_summary_{horizon}.png",
                    dpi=300,
                    bbox_inches="tight",
                )
                plt.show()
                plt.close()

                # 2. Bar Plot (Feature Importance)
                plt.figure(figsize=(10, 8), dpi=300)
                shap.summary_plot(
                    shap_values, X_sample, plot_type="bar", show=False, max_display=20
                )
                plt.title(
                    f"{horizon} Glucose Prediction - SHAP Feature Importance",
                    pad=20,
                    fontsize=14,
                    fontweight="bold",
                )
                plt.tight_layout()
                plt.savefig(
                    f"{self.results_dir}/shap_importance_{horizon}.png",
                    dpi=300,
                    bbox_inches="tight",
                )
                plt.show()
                plt.close()

                # 3. **FIXED** Waterfall Plot
                plt.figure(figsize=(12, 8), dpi=300)
                shap_explanation = shap.Explanation(
                    values=shap_values[0],
                    base_values=explainer.expected_value,
                    data=X_sample.iloc[0].values,  # ✅ 需要 .values
                    feature_names=X_sample.columns.tolist(),
                )
                # ✅ 使用正确的API
                shap.waterfall_plot(shap_explanation, max_display=15, show=False)
                plt.title(
                    f"{horizon} Glucose Prediction - SHAP Waterfall Plot (Sample 1)",
                    pad=20,
                    fontsize=14,
                    fontweight="bold",
                )
                plt.tight_layout()
                plt.savefig(
                    f"{self.results_dir}/shap_waterfall_{horizon}.png",
                    dpi=300,
                    bbox_inches="tight",
                )
                plt.show()
                plt.close()

                # 4. Calculate and save feature importance
                feature_importance = pd.DataFrame(
                    {
                        "feature": X_sample.columns,
                        "importance": np.abs(shap_values).mean(axis=0),
                    }
                ).sort_values("importance", ascending=False)

                logger.info(f"\n{horizon} prediction - Top 15 important features:")
                # ✅ 使用 _ 忽略索引
                for _, row in feature_importance.head(15).iterrows():
                    logger.info(f"  {row['feature']}: {row['importance']:.4f}")

                # Save feature importance
                feature_importance.to_csv(
                    f"{self.results_dir}/feature_importance_{horizon}.csv", index=False
                )

                logger.info(f"\n{horizon} SHAP analysis completed")

            except Exception as e:
                logger.error(f"SHAP analysis failed ({horizon}): {e}")
                traceback.print_exc()
                continue


def run_complete_pipeline(
    n_trials: int = 200, shap_samples: int = 100
) -> Tuple[RandomForestGlucosePredictor, Dict]:
    """Run complete prediction pipeline"""
    logger.info("=" * 60)
    logger.info(
        "RandomForest Glucose Prediction Pipeline - Bayesian Optimization Version"
    )
    logger.info("=" * 60)

    try:
        predictor = RandomForestGlucosePredictor(random_state=RANDOM_SEED)
        train_df, val_df, test_df = predictor.load_data()
        predictor.train_models(train_df, val_df, n_trials=n_trials)
        results = predictor.evaluate_on_test(test_df)
        predictor.plot_predictions(results)
        predictor.plot_time_series_by_patient(results, test_df, num_patients=2)
        predictor.shap_analysis(results, max_samples=shap_samples)
        generate_summary_report(predictor, results)

        logger.info("\n" + "=" * 60)
        logger.info("Pipeline execution completed!")
        logger.info(f"All results saved to: {predictor.results_dir}/")
        logger.info("=" * 60)

        return predictor, results

    except Exception as e:
        logger.error(f"Pipeline execution failed: {e}")
        traceback.print_exc()
        raise


def generate_summary_report(
    predictor: RandomForestGlucosePredictor, results: Dict
) -> None:
    """Generate summary report"""
    report_path = f"{predictor.results_dir}/summary_report.txt"

    with open(report_path, "w", encoding="utf-8") as f:
        f.write("=" * 60 + "\n")
        f.write("RandomForest Glucose Prediction Model - Summary Report\n")
        f.write("(Bayesian Optimization Version)\n")
        f.write("=" * 60 + "\n\n")

        f.write("1. Model Performance\n")
        f.write("-" * 40 + "\n")
        for horizon, data in results.items():
            f.write(f"\n{horizon} prediction:\n")
            f.write(f"  RMSE: {data['rmse']:.4f} mg/dL\n")
            f.write(f"  MAE:  {data['mae']:.4f} mg/dL\n")
            f.write(f"  R²:   {data['r2']:.4f}\n")

        f.write("\n\n2. Best Hyperparameters (Bayesian Optimization)\n")
        f.write("-" * 40 + "\n")

        f.write("\n30-minute model:\n")
        for param, value in predictor.best_params_30min.items():
            f.write(f"  {param}: {value}\n")

        f.write("\n60-minute model:\n")
        for param, value in predictor.best_params_60min.items():
            f.write(f"  {param}: {value}\n")

        f.write("\n\n3. Number of Features\n")
        f.write("-" * 40 + "\n")
        f.write(f"Total features: {len(predictor.feature_names)}\n")

        f.write("\n\n4. Output Files List\n")
        f.write("-" * 40 + "\n")
        f.write("  - best_hyperparameters.csv (Best hyperparameters)\n")
        f.write("  - optimization_history_30min.csv (30-min optimization history)\n")
        f.write("  - optimization_history_60min.csv (60-min optimization history)\n")
        f.write("  - optimization_history.png (Optimization convergence plot)\n")
        f.write("  - predictions_30min.csv (30-minute prediction results)\n")
        f.write("  - predictions_60min.csv (60-minute prediction results)\n")
        f.write("  - feature_importance_30min.csv (30-minute feature importance)\n")
        f.write("  - feature_importance_60min.csv (60-minute feature importance)\n")
        f.write(
            "  - prediction_comparison_30min.png (30-minute prediction comparison)\n"
        )
        f.write(
            "  - prediction_comparison_60min.png (60-minute prediction comparison)\n"
        )
        f.write("  - shap_summary_30min.png (30-minute SHAP summary)\n")
        f.write("  - shap_summary_60min.png (60-minute SHAP summary)\n")
        f.write("  - shap_importance_30min.png (30-minute SHAP importance)\n")
        f.write("  - shap_importance_60min.png (60-minute SHAP importance)\n")
        f.write("  - shap_waterfall_30min.png (30-minute SHAP waterfall)\n")
        f.write("  - shap_waterfall_60min.png (60-minute SHAP waterfall)\n")

        f.write("\n\n5. Model Interpretation\n")
        f.write("-" * 40 + "\n")
        f.write("RMSE (Root Mean Square Error): Lower is better\n")
        f.write("MAE (Mean Absolute Error): Lower is better\n")
        f.write("R² (R-squared): Range 0-1, closer to 1 is better\n")
        f.write("\nClinical glucose reference ranges:\n")
        f.write("  - Normal range: 70-180 mg/dL\n")
        f.write("  - Hypoglycemia: < 70 mg/dL\n")
        f.write("  - Hyperglycemia: > 180 mg/dL\n")

        f.write("\n\n6. Optimization Method & Model Type\n")
        f.write("-" * 40 + "\n")
        f.write("Method: Bayesian Optimization (Optuna TPESampler)\n")
        f.write("Model: RandomForest Regressor\n")
        f.write("Algorithm: Ensemble of Decision Trees\n")
        f.write("\nAdvantages:\n")
        f.write("  - Robust to outliers and non-linear relationships\n")
        f.write("  - No need for feature scaling\n")
        f.write("  - Built-in feature importance calculation\n")
        f.write("  - Good generalization capability\n")
        f.write("  - Parallelizable training process\n")
        f.write("\nBayesian Optimization benefits:\n")
        f.write("  - More efficient than random search\n")
        f.write("  - Automatically balances exploration and exploitation\n")
        f.write("  - Finds optimal hyperparameters faster\n")

        f.write("\n" + "=" * 60 + "\n")

    logger.info(f"Summary report saved to: {report_path}")


if __name__ == "__main__":
    """
    Usage instructions:
    1. Ensure data files are in the current directory:
       - train_enhanced_processed.csv
       - val_enhanced_processed.csv
       - test_enhanced_processed.csv

    2. Adjust parameters:
       - n_trials: Number of Bayesian optimization trials (default 200, can increase to 500+)
       - shap_samples: SHAP analysis sample size (default 100)

    3. Run the code:
       python script_name.py

    4. Results are saved in the results/ directory

    Note: This FIXED version resolves:
          - SHAP waterfall plot API issue
          - Patient time series index matching bug
          - Bootstrap/max_samples compatibility
    """

    N_TRIALS = 500  # ✅ 可根据计算资源调整
    SHAP_SAMPLES = 100

    try:
        predictor, results = run_complete_pipeline(
            n_trials=N_TRIALS, shap_samples=SHAP_SAMPLES
        )

        print("\n" + "=" * 60)
        print("Execution completed! Main results:")
        print("=" * 60)
        print("\nTest set performance:")
        for horizon, data in results.items():
            print(f"\n{horizon}:")
            print(f"  RMSE: {data['rmse']:.4f} mg/dL")
            print(f"  MAE:  {data['mae']:.4f} mg/dL")
            print(f"  R²:   {data['r2']:.4f}")

        print(f"\nAll results saved to: {predictor.results_dir}/")
        print("\nMain files:")
        print("  - summary_report.txt (Summary report)")
        print("  - optimization_history.png (Bayesian optimization convergence)")
        print("  - prediction_comparison_*.png (Prediction comparison plots)")
        print("  - shap_*.png (SHAP interpretability plots)")
        print("  - predictions_*.csv (Prediction results)")
        print("  - feature_importance_*.csv (Feature importance)")

    except Exception as e:
        logger.error(f"Execution failed: {e}")
        traceback.print_exc()

In [ ]:
def plot_predictions(
    results: Dict[str, Dict[str, Any]], sample_points: int = 600
) -> None:
    """
    Plot prediction comparison (4-in-1 plot)

    Parameters:
    -----------
    results : Dict[str, Dict[str, Any]]
        包含预测结果的字典
    sample_points : int
        要绘制的前N个时间点数量(默认300)
    """
    logger.info("\nGenerating prediction comparison plots...")

    for horizon, data in results.items():
        y_true = data["y_true"]
        y_pred = data["y_pred"]

        # 直接取前 sample_points 个点,而不是采样
        plot_samples = min(sample_points, len(y_true))
        y_true_plot = y_true[:plot_samples]
        y_pred_plot = y_pred[:plot_samples]
        indices = np.arange(plot_samples)  # 0, 1, 2, ..., plot_samples-1

        fig, axes = plt.subplots(2, 2, figsize=(16, 12), dpi=300)

        # 1. Time series comparison
        axes[0, 0].plot(
            indices,
            y_true_plot,
            "b-",
            label="Ground Truth",
            alpha=0.7,
            linewidth=1.5,
        )
        axes[0, 0].plot(
            indices,
            y_pred_plot,
            "r--",
            label="Prediction",
            alpha=0.7,
            linewidth=1.5,
        )
        axes[0, 0].fill_between(
            indices, y_true_plot, y_pred_plot, alpha=0.2, color="gray"
        )
        axes[0, 0].set_xlabel("Time Point (5-min intervals)", fontsize=11)
        axes[0, 0].set_ylabel("Glucose Level (mg/dL)", fontsize=11)
        axes[0, 0].set_title(
            f"{horizon} Glucose Prediction - Time Series Comparison (First {plot_samples} points)",
            fontsize=13,
            fontweight="bold",
        )
        axes[0, 0].legend(fontsize=10)
        axes[0, 0].grid(True, alpha=0.3)

        # 2. Scatter plot (使用全部数据点,保持原有逻辑)
        axes[0, 1].scatter(y_true, y_pred, alpha=0.3, s=10, c="steelblue")

        # Add perfect prediction line
        min_val = min(y_true.min(), y_pred.min())
        max_val = max(y_true.max(), y_pred.max())
        axes[0, 1].plot(
            [min_val, max_val],
            [min_val, max_val],
            "r--",
            linewidth=2,
            label="Perfect Prediction",
            alpha=0.8,
        )

        axes[0, 1].set_xlabel("True Glucose Level (mg/dL)", fontsize=11)
        axes[0, 1].set_ylabel("Predicted Glucose Level (mg/dL)", fontsize=11)
        axes[0, 1].set_title(
            f"{horizon} Glucose Prediction - Scatter Plot",
            fontsize=13,
            fontweight="bold",
        )
        axes[0, 1].legend(fontsize=10)
        axes[0, 1].grid(True, alpha=0.3)

        # Add performance metrics
        textstr = f"RMSE: {data['rmse']:.2f} mg/dL\nMAE: {data['mae']:.2f} mg/dL\nR²: {data['r2']:.4f}"
        axes[0, 1].text(
            0.05,
            0.95,
            textstr,
            transform=axes[0, 1].transAxes,
            verticalalignment="top",
            bbox=dict(boxstyle="round", facecolor="wheat", alpha=0.7),
            fontsize=10,
        )

        # 3. Residual plot (使用全部数据点)
        residuals = y_true - y_pred
        axes[1, 0].scatter(y_pred, residuals, alpha=0.3, s=10, c="coral")
        axes[1, 0].axhline(y=0, color="r", linestyle="--", linewidth=2)
        axes[1, 0].set_xlabel("Predicted Glucose Level (mg/dL)", fontsize=11)
        axes[1, 0].set_ylabel("Residual (mg/dL)", fontsize=11)
        axes[1, 0].set_title(
            f"{horizon} Glucose Prediction - Residual Plot",
            fontsize=13,
            fontweight="bold",
        )
        axes[1, 0].grid(True, alpha=0.3)

        # Add residual statistics
        residual_mean = np.mean(residuals)
        residual_std = np.std(residuals)
        axes[1, 0].text(
            0.05,
            0.95,
            f"Mean: {residual_mean:.2f}\nStd Dev: {residual_std:.2f}",
            transform=axes[1, 0].transAxes,
            verticalalignment="top",
            bbox=dict(boxstyle="round", facecolor="lightblue", alpha=0.7),
            fontsize=10,
        )

        # 4. Residual distribution (使用全部数据点)
        axes[1, 1].hist(residuals, bins=50, alpha=0.7, color="blue", edgecolor="black")
        axes[1, 1].axvline(
            x=0, color="r", linestyle="--", linewidth=2, label="Zero Residual"
        )

        # Add normal distribution fit curve
        mu, sigma = stats.norm.fit(residuals)
        x = np.linspace(residuals.min(), residuals.max(), 100)
        p = stats.norm.pdf(x, mu, sigma)
        ax2 = axes[1, 1].twinx()
        ax2.plot(x, p, "r-", linewidth=2, label="Normal Distribution Fit")
        ax2.set_ylabel("Probability Density", fontsize=10)

        axes[1, 1].set_xlabel("Residual (mg/dL)", fontsize=11)
        axes[1, 1].set_ylabel("Frequency", fontsize=11)
        axes[1, 1].set_title(
            f"{horizon} Glucose Prediction - Residual Distribution",
            fontsize=13,
            fontweight="bold",
        )
        axes[1, 1].legend(loc="upper left", fontsize=9)
        ax2.legend(loc="upper right", fontsize=9)
        axes[1, 1].grid(True, alpha=0.3)

        plt.tight_layout()
        plt.show()
        plt.close()

        logger.info(f"{horizon} prediction curves saved")


plot_predictions(results)

In [ ]:
def plot_predictions(
    results: Dict[str, Dict[str, Any]], sample_points: int = 600
) -> None:
    """
    绘制预测曲线对比图

    Parameters:
    -----------
    results : Dict[str, Dict[str, Any]]
        包含预测结果的字典
    sample_points : int
        要绘制的前N个时间点数量(默认300)
    """
    logger.info("\nGenerating prediction comparison plots...")

    for horizon, data in results.items():
        y_true = data["y_true"]
        y_pred = data["y_pred"]

        # 直接取前 sample_points 个点
        plot_samples = min(sample_points, len(y_true))
        y_true_plot = y_true[:plot_samples]
        y_pred_plot = y_pred[:plot_samples]
        indices = np.arange(plot_samples)

        # 创建单个图表
        plt.figure(figsize=(18, 6), dpi=300)

        # 绘制真实值
        plt.plot(
            indices,
            y_true_plot,
            label="True Glucose (mg/dL)",
            color="black",
            linewidth=2,
            alpha=0.8,
        )

        # 绘制预测值
        plt.plot(
            indices,
            y_pred_plot,
            label="Predicted Glucose (mg/dL)",
            color="red",
            linestyle="--",
            linewidth=1.5,
            alpha=0.8,
        )

        # 填充区域
        plt.fill_between(indices, y_true_plot, y_pred_plot, alpha=0.2, color="gray")

        plt.title(
            f"Glucose Prediction Comparison on Test Set - {horizon} (First {plot_samples} points)",
            fontsize=14,
            fontweight="bold",
        )
        plt.xlabel("Time (5-min intervals)", fontsize=12)
        plt.ylabel("Glucose (mg/dL)", fontsize=12)
        plt.legend(loc="upper right", fontsize=11)
        plt.grid(True, alpha=0.3)
        plt.margins(x=0.02, y=0.1)
        plt.tight_layout()
        plt.show()
        plt.close()

        logger.info(f"{horizon} prediction curve saved")


plot_predictions(results)

In [ ]:
import matplotlib.dates as mdates
import numpy as np
import pandas as pd
import logging
import traceback

logger = logging.getLogger(__name__)


def plot_time_series_by_patient_standalone(
    results: Dict[str, Dict[str, Any]],
    test_df: pd.DataFrame,
    num_patients: int = 6,
    results_dir: str = "results",
) -> None:
    """
    绘制每个患者的血糖时间序列图(独立函数版本)

    Parameters:
    -----------
    results : Dict[str, Dict[str, Any]]
        预测结果字典,包含'30min'和'60min'键
    test_df : pd.DataFrame
        测试数据集
    num_patients : int
        绘制的患者数量
    results_dir : str
        保存结果的目录
    """
    if "timestamp" not in test_df.columns or "patient_id" not in test_df.columns:
        logger.warning("Missing timestamp or patient_id columns")
        return

    logger.info("\nGenerating patient-level time series plots...")

    # 确保时间戳是datetime类型
    test_df = test_df.copy()
    test_df["timestamp"] = pd.to_datetime(test_df["timestamp"])

    patient_ids = sorted(test_df["patient_id"].unique())[:num_patients]

    for horizon, data in results.items():
        target_col = "glucose_30min" if horizon == "30min" else "glucose_60min"

        for patient_id in patient_ids:
            try:
                # 获取该患者的数据
                patient_mask = test_df["patient_id"] == patient_id
                patient_data = test_df[patient_mask].copy()

                if len(patient_data) < 10:
                    logger.info(
                        f"Patient {patient_id}: insufficient data (< 10 points)"
                    )
                    continue

                # 移除NaN值
                patient_data = patient_data.dropna(subset=[target_col])

                if len(patient_data) == 0:
                    logger.info(
                        f"Patient {patient_id}: no valid data after removing NaN"
                    )
                    continue

                # 重建索引以便对齐
                patient_data = patient_data.reset_index(drop=False)
                original_indices = patient_data["index"].values

                # 获取对应的预测值
                y_pred_patient = []
                y_true_patient = []
                timestamps = []

                for idx in original_indices:
                    if idx in data["y_true"].index:
                        pred_idx = data["y_true"].index.get_loc(idx)
                        y_pred_patient.append(data["y_pred"][pred_idx])
                        y_true_patient.append(data["y_true"].iloc[pred_idx])
                        timestamps.append(
                            patient_data[patient_data["index"] == idx][
                                "timestamp"
                            ].iloc[0]
                        )

                if len(y_pred_patient) < 10:
                    logger.info(
                        f"Patient {patient_id}: insufficient matched predictions (< 10 points)"
                    )
                    continue

                y_pred_patient = np.array(y_pred_patient)
                y_true_patient = np.array(y_true_patient)
                timestamps = pd.to_datetime(timestamps)

                # 绘图
                fig, ax = plt.subplots(figsize=(16, 6), dpi=300)

                ax.plot(
                    timestamps,
                    y_true_patient,
                    "b-",
                    label="Ground Truth",
                    linewidth=2,
                    alpha=0.7,
                )
                ax.plot(
                    timestamps,
                    y_pred_patient,
                    "r--",
                    label="Prediction",
                    linewidth=2,
                    alpha=0.7,
                )
                ax.fill_between(
                    timestamps,
                    y_true_patient,
                    y_pred_patient,
                    alpha=0.2,
                    color="gray",
                )

                # 添加目标区域
                ax.axhspan(
                    70,
                    180,
                    alpha=0.1,
                    color="green",
                    label="Target Range (70-180 mg/dL)",
                )

                # 设置x轴格式
                time_range = timestamps.max() - timestamps.min()

                if time_range.days > 7:
                    ax.xaxis.set_major_locator(
                        mdates.DayLocator(interval=max(1, time_range.days // 10))
                    )
                    ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m-%d"))
                elif time_range.days > 1:
                    ax.xaxis.set_major_locator(mdates.HourLocator(interval=6))
                    ax.xaxis.set_major_formatter(mdates.DateFormatter("%m-%d %H:%M"))
                else:
                    ax.xaxis.set_major_locator(mdates.HourLocator(interval=3))
                    ax.xaxis.set_major_formatter(mdates.DateFormatter("%H:%M"))

                plt.xticks(rotation=45)

                ax.set_xlabel("Time", fontsize=12)
                ax.set_ylabel("Glucose Level (mg/dL)", fontsize=12)
                ax.set_title(
                    f"{horizon} Glucose Prediction - Patient {patient_id} Time Series\n"
                    f"(n={len(y_true_patient)} time points)",
                    fontsize=14,
                    fontweight="bold",
                    pad=20,
                )

                ax.legend(fontsize=10, loc="best")
                ax.grid(True, alpha=0.3)

                plt.tight_layout()
                plt.savefig(
                    f"{results_dir}/time_series_{horizon}_patient_{patient_id}.png",
                    dpi=300,
                    bbox_inches="tight",
                )
                plt.show()
                plt.close()

                logger.info(
                    f"Patient {patient_id} ({horizon}): plotted {len(y_true_patient)} time points"
                )

            except Exception as e:
                logger.warning(
                    f"Failed to plot {horizon} for patient {patient_id}: {e}"
                )
                traceback.print_exc()
                continue

In [ ]:
def calculate_patient_metrics_standalone(
    results: Dict[str, Dict[str, Any]],
    test_df: pd.DataFrame,
    results_dir: str = "results",
) -> pd.DataFrame:
    """
    计算每个患者的预测性能指标(RMSE, MAE, R²)

    Parameters:
    -----------
    results : Dict
        预测结果字典
    test_df : pd.DataFrame
        测试数据集
    results_dir : str
        保存结果的目录

    Returns:
    --------
    pd.DataFrame: 包含每个患者性能指标的数据框
    """
    logger.info("\nCalculating per-patient metrics...")

    if "patient_id" not in test_df.columns:
        logger.warning("Missing patient_id column")
        return pd.DataFrame()

    test_df = test_df.copy()
    patient_ids = sorted(test_df["patient_id"].unique())

    all_metrics = []

    for horizon, data in results.items():
        target_col = "glucose_30min" if horizon == "30min" else "glucose_60min"

        for patient_id in patient_ids:
            try:
                # 获取患者数据
                patient_mask = test_df["patient_id"] == patient_id
                patient_data = test_df[patient_mask].copy()
                patient_data = patient_data.dropna(subset=[target_col])

                if len(patient_data) < 5:
                    continue

                # 重建索引对齐
                patient_data = patient_data.reset_index(drop=False)
                original_indices = patient_data["index"].values

                # 获取预测值和真实值
                y_pred = []
                y_true = []

                for idx in original_indices:
                    if idx in data["y_true"].index:
                        pred_idx = data["y_true"].index.get_loc(idx)
                        y_pred.append(data["y_pred"][pred_idx])
                        y_true.append(data["y_true"].iloc[pred_idx])

                if len(y_pred) < 5:
                    continue

                y_pred = np.array(y_pred)
                y_true = np.array(y_true)

                # 计算指标
                rmse = np.sqrt(mean_squared_error(y_true, y_pred))
                mae = mean_absolute_error(y_true, y_pred)
                r2 = r2_score(y_true, y_pred)

                all_metrics.append(
                    {
                        "patient_id": patient_id,
                        "horizon": horizon,
                        "n_samples": len(y_true),
                        "RMSE": rmse,
                        "MAE": mae,
                        "R2": r2,
                    }
                )

                logger.info(
                    f"Patient {patient_id} ({horizon}): "
                    f"RMSE={rmse:.2f}, MAE={mae:.2f}, R²={r2:.4f}, n={len(y_true)}"
                )

            except Exception as e:
                logger.warning(
                    f"Failed to calculate metrics for patient {patient_id} ({horizon}): {e}"
                )
                continue

    metrics_df = pd.DataFrame(all_metrics)

    # 保存结果
    if len(metrics_df) > 0:
        metrics_df.to_csv(f"{results_dir}/patient_metrics.csv", index=False)
        logger.info(f"\nPatient metrics saved to {results_dir}/patient_metrics.csv")

        # 打印汇总统计
        logger.info("\nSummary statistics across all patients:")
        for horizon in metrics_df["horizon"].unique():
            horizon_data = metrics_df[metrics_df["horizon"] == horizon]
            logger.info(f"\n{horizon}:")
            logger.info(
                f"  RMSE: {horizon_data['RMSE'].mean():.2f} ± {horizon_data['RMSE'].std():.2f}"
            )
            logger.info(
                f"  MAE:  {horizon_data['MAE'].mean():.2f} ± {horizon_data['MAE'].std():.2f}"
            )
            logger.info(
                f"  R²:   {horizon_data['R2'].mean():.4f} ± {horizon_data['R2'].std():.4f}"
            )

    return metrics_df


In [ ]:
def plot_patient_first_n_points_standalone(
    results: Dict[str, Dict[str, Any]],
    test_df: pd.DataFrame,
    n_points: int = 600,
    highlight_ranges: bool = False,
    num_patients: int = None,
    results_dir: str = "results",
) -> None:
    """
    绘制每个患者前N个时间点的真实血糖与预测血糖(独立函数版本)

    Parameters:
    -----------
    results : Dict
        预测结果字典
    test_df : pd.DataFrame
        测试数据
    n_points : int
        绘制的时间点数量(默认600)
    highlight_ranges : bool
        是否标注低血糖和高血糖区域(默认False)
    num_patients : int
        绘制的患者数量(None表示全部)
    results_dir : str
        保存结果的目录
    """
    logger.info(f"\nGenerating first {n_points} points plots for patients...")

    if "patient_id" not in test_df.columns:
        logger.warning("Missing patient_id column")
        return

    test_df = test_df.copy()
    test_df["timestamp"] = pd.to_datetime(test_df["timestamp"])

    patient_ids = sorted(test_df["patient_id"].unique())
    if num_patients is not None:
        patient_ids = patient_ids[:num_patients]

    for horizon, data in results.items():
        target_col = "glucose_30min" if horizon == "30min" else "glucose_60min"

        for patient_id in patient_ids:
            try:
                # 获取患者数据
                patient_mask = test_df["patient_id"] == patient_id
                patient_data = test_df[patient_mask].copy()
                patient_data = patient_data.dropna(subset=[target_col])

                if len(patient_data) == 0:
                    continue

                # 重建索引对齐
                patient_data = patient_data.reset_index(drop=False)
                original_indices = patient_data["index"].values

                # 获取预测值和真实值
                y_pred_list = []
                y_true_list = []
                timestamps_list = []

                for idx in original_indices:
                    if idx in data["y_true"].index:
                        pred_idx = data["y_true"].index.get_loc(idx)
                        y_pred_list.append(data["y_pred"][pred_idx])
                        y_true_list.append(data["y_true"].iloc[pred_idx])
                        timestamps_list.append(
                            patient_data[patient_data["index"] == idx][
                                "timestamp"
                            ].iloc[0]
                        )

                if len(y_pred_list) == 0:
                    continue

                # 截取前n_points个点
                plot_length = min(n_points, len(y_pred_list))
                y_pred_plot = np.array(y_pred_list[:plot_length])
                y_true_plot = np.array(y_true_list[:plot_length])
                timestamps_plot = pd.to_datetime(timestamps_list[:plot_length])
                time_indices = np.arange(plot_length)

                # 计算该患者的指标
                rmse = np.sqrt(mean_squared_error(y_true_plot, y_pred_plot))
                mae = mean_absolute_error(y_true_plot, y_pred_plot)
                r2 = r2_score(y_true_plot, y_pred_plot)

                # 绘图
                fig, ax = plt.subplots(figsize=(18, 6), dpi=300)

                # 如果需要标注血糖范围
                if highlight_ranges:
                    ax.axhspan(
                        0, 70, alpha=0.15, color="red", label="Hypoglycemia (<70 mg/dL)"
                    )
                    ax.axhspan(
                        70,
                        180,
                        alpha=0.1,
                        color="green",
                        label="Target Range (70-180 mg/dL)",
                    )
                    ax.axhspan(
                        180,
                        400,
                        alpha=0.15,
                        color="orange",
                        label="Hyperglycemia (>180 mg/dL)",
                    )

                # 绘制真实值和预测值
                ax.plot(
                    time_indices,
                    y_true_plot,
                    label="True Glucose (mg/dL)",
                    color="black",
                    linewidth=2,
                    alpha=0.8,
                )
                ax.plot(
                    time_indices,
                    y_pred_plot,
                    label="Predicted Glucose (mg/dL)",
                    color="red",
                    linestyle="--",
                    linewidth=1.5,
                    alpha=0.8,
                )

                # 填充区域
                ax.fill_between(
                    time_indices, y_true_plot, y_pred_plot, alpha=0.2, color="gray"
                )

                # 设置标题和标签
                ax.set_title(
                    f"Patient {patient_id} - {horizon} Glucose Prediction (First {plot_length} points)\n"
                    f"RMSE: {rmse:.2f} mg/dL, MAE: {mae:.2f} mg/dL, R²: {r2:.4f}",
                    fontsize=14,
                    fontweight="bold",
                    pad=20,
                )
                ax.set_xlabel("Time Point (5-min intervals)", fontsize=12)
                ax.set_ylabel("Glucose (mg/dL)", fontsize=12)
                ax.legend(loc="best", fontsize=11)
                ax.grid(True, alpha=0.3)
                ax.set_ylim(
                    [
                        min(y_true_plot.min(), y_pred_plot.min()) - 20,
                        max(y_true_plot.max(), y_pred_plot.max()) + 20,
                    ]
                )

                plt.tight_layout()
                plt.savefig(
                    f"{results_dir}/first_{n_points}pts_{horizon}_patient_{patient_id}.png",
                    dpi=300,
                    bbox_inches="tight",
                )
                plt.show()
                plt.close()

                logger.info(
                    f"Patient {patient_id} ({horizon}): "
                    f"plotted {plot_length} points, RMSE={rmse:.2f}, MAE={mae:.2f}, R²={r2:.4f}"
                )

            except Exception as e:
                logger.warning(
                    f"Failed to plot first {n_points} points for patient {patient_id} ({horizon}): {e}"
                )
                traceback.print_exc()
                continue


In [ ]:
# 加载测试数据
test_df = pd.read_csv("test_enhanced_processed.csv")

In [ ]:
# ===== 1. 绘制患者时间序列图 =====
plot_time_series_by_patient_standalone(
    results=results, test_df=test_df, num_patients=6, results_dir="results"
)

In [ ]:
# ===== 2. 计算每个患者的性能指标 =====
patient_metrics = calculate_patient_metrics_standalone(
    results=results, test_df=test_df, results_dir="results"
)

# 查看结果
print("\n患者性能指标:")
print(patient_metrics.head(12))

In [ ]:
# 按患者分组查看
print("\n按患者查看30分钟预测:")
print(patient_metrics[patient_metrics["horizon"] == "30min"])

In [ ]:
# 按患者分组查看
print("\n按患者查看60分钟预测:")
print(patient_metrics[patient_metrics["horizon"] == "60min"])

In [ ]:
# ===== 3. 绘制前600个时间点(不标注血糖范围) =====
plot_patient_first_n_points_standalone(
    results=results,
    test_df=test_df,
    n_points=600,
    highlight_ranges=False,
    num_patients=None,  # 绘制所有患者
    results_dir="results",
)

In [ ]:
# ===== 4. 绘制前600个时间点(标注血糖范围,只绘制前3个患者) =====
plot_patient_first_n_points_standalone(
    results=results,
    test_df=test_df,
    n_points=600,
    highlight_ranges=True,
    num_patients=3,
    results_dir="results",
)

In [ ]:
# ===== 5. 绘制前300个时间点(快速预览) =====
plot_patient_first_n_points_standalone(
    results=results,
    test_df=test_df,
    n_points=300,
    highlight_ranges=False,
    num_patients=5,
    results_dir="results",
)